# 6D Hénon-Heiles Ground State with LP+PINN

This notebook computes the zero-point energy of the 6D Hénon-Heiles system using a Lagrangian Physics-Informed Neural Network.

**Before running:**
1. Go to Runtime → Change runtime type → Select **T4 GPU**
2. Click Connect in the top right

**Expected results:**
- Final energy: ~2.97 a.u. (within 2% of target)
- Below harmonic ZPE of 3.0 a.u.
- Training time: ~20 hours on T4

In [ ]:
# Verify GPU is available
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Mount Google Drive for checkpoints (optional but recommended for long training)
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/henon_heiles_checkpoints'
RESULTS_DIR = '/content/drive/MyDrive/henon_heiles_results'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

In [ ]:
import json
import time
from datetime import datetime
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

#=============================================================================
# CONFIGURATION
#=============================================================================

class Config:
    """Training configuration."""
    # Network architecture
    input_dim = 6
    output_dim = 12  # 6 real + 6 imaginary momentum components
    hidden_layers = 8
    hidden_dim = 512
    activation = "tanh"

    # Physics parameters
    hbar = 1.0  # Natural units
    mass = 1.0
    lambda_hh = 1.0 / np.sqrt(80)  # ≈ 0.111803

    # Training parameters
    learning_rate = 1e-4
    n_collocation = 100_000
    max_epochs = 50_000
    checkpoint_interval = 1000
    log_interval = 100

    # Domain (sampling range for each coordinate)
    domain_min = -3.0
    domain_max = 3.0

    # Loss weights
    weight_physics = 1.0
    weight_curl = 0.1

    # Initial energy guess (harmonic ZPE = n*hbar*omega/2 = 6*1*1/2 = 3.0)
    initial_energy = 3.0

config = Config()
print(f"Lambda (Hénon-Heiles coupling): {config.lambda_hh:.6f}")
print(f"Network: {config.hidden_layers} layers x {config.hidden_dim} neurons")
print(f"Collocation points: {config.n_collocation:,}")

In [ ]:
#=============================================================================
# NETWORK ARCHITECTURE
#=============================================================================

class LagrangianPINN(nn.Module):
    """
    Lagrangian Physics-Informed Neural Network for quantum mechanics.
    Outputs complex momentum field p(x) = p_real(x) + i*p_imag(x).
    """

    def __init__(self, config):
        super().__init__()
        self.config = config

        layers = []
        layers.append(nn.Linear(config.input_dim, config.hidden_dim))
        layers.append(nn.Tanh())

        for _ in range(config.hidden_layers - 1):
            layers.append(nn.Linear(config.hidden_dim, config.hidden_dim))
            layers.append(nn.Tanh())

        layers.append(nn.Linear(config.hidden_dim, config.output_dim))
        self.network = nn.Sequential(*layers)

        # Learnable energy parameter
        self.log_energy = nn.Parameter(torch.tensor(np.log(config.initial_energy)))
        self._init_weights()

    def _init_weights(self):
        for module in self.network.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    @property
    def energy(self):
        return torch.exp(self.log_energy)

    def forward(self, x):
        output = self.network(x)
        p_real = output[:, :6]
        p_imag = output[:, 6:]
        return p_real, p_imag

# Count parameters
model = LagrangianPINN(config)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")

In [ ]:
#=============================================================================
# PHYSICS FUNCTIONS
#=============================================================================

def henon_heiles_potential(x, lambda_hh):
    """
    6D Hénon-Heiles potential with periodic coupling.
    V(x) = ½Σᵢxᵢ² + λΣᵢ(xᵢ²xᵢ₊₁ - xᵢ₊₁³/3)
    """
    V_harmonic = 0.5 * torch.sum(x**2, dim=1)
    V_anharmonic = torch.zeros_like(V_harmonic)
    for i in range(6):
        j = (i + 1) % 6  # Periodic: x₇ = x₁
        xi = x[:, i]
        xj = x[:, j]
        V_anharmonic += xi**2 * xj - xj**3 / 3
    return V_harmonic + lambda_hh * V_anharmonic


def compute_divergence(p, x):
    """Compute divergence ∇·p."""
    div_p = torch.zeros(x.shape[0], device=x.device)
    for i in range(x.shape[1]):
        grad_pi = torch.autograd.grad(
            outputs=p[:, i],
            inputs=x,
            grad_outputs=torch.ones_like(p[:, i]),
            create_graph=True,
            retain_graph=True
        )[0]
        div_p += grad_pi[:, i]
    return div_p


def compute_curl_loss(p_real, p_imag, x):
    """Compute curl loss for irrotationality constraint."""
    dim = x.shape[1]
    curl_loss = torch.tensor(0.0, device=x.device)

    # Real part gradients
    grad_p_real = []
    for i in range(dim):
        grad_pi = torch.autograd.grad(
            outputs=p_real[:, i],
            inputs=x,
            grad_outputs=torch.ones_like(p_real[:, i]),
            create_graph=True,
            retain_graph=True
        )[0]
        grad_p_real.append(grad_pi)

    for i in range(dim):
        for j in range(i + 1, dim):
            diff = grad_p_real[i][:, j] - grad_p_real[j][:, i]
            curl_loss += torch.mean(diff**2)

    # Imaginary part gradients
    grad_p_imag = []
    for i in range(dim):
        grad_pi = torch.autograd.grad(
            outputs=p_imag[:, i],
            inputs=x,
            grad_outputs=torch.ones_like(p_imag[:, i]),
            create_graph=True,
            retain_graph=True
        )[0]
        grad_p_imag.append(grad_pi)

    for i in range(dim):
        for j in range(i + 1, dim):
            diff = grad_p_imag[i][:, j] - grad_p_imag[j][:, i]
            curl_loss += torch.mean(diff**2)

    return curl_loss


def compute_physics_loss(model, x, config):
    """Compute QHJE physics loss."""
    x.requires_grad_(True)
    p_real, p_imag = model(x)
    V = henon_heiles_potential(x, config.lambda_hh)

    div_p_real = compute_divergence(p_real, x)
    div_p_imag = compute_divergence(p_imag, x)

    p_real_sq = torch.sum(p_real**2, dim=1)
    p_imag_sq = torch.sum(p_imag**2, dim=1)
    p_cross = torch.sum(p_real * p_imag, dim=1)

    E = model.energy

    # Real: (p_r² - p_i²)/2 + V - (ℏ/2)∇·p_i = E
    real_residual = (p_real_sq - p_imag_sq) / 2 + V - 0.5 * config.hbar * div_p_imag - E
    # Imaginary: p_r·p_i + (ℏ/2)∇·p_r = 0
    imag_residual = p_cross + 0.5 * config.hbar * div_p_real

    physics_loss = torch.mean(real_residual**2) + torch.mean(imag_residual**2)
    return physics_loss, E

In [ ]:
#=============================================================================
# TRAINING FUNCTIONS
#=============================================================================

def sample_collocation_points(n_points, config, device):
    x = torch.rand(n_points, config.input_dim, device=device)
    x = x * (config.domain_max - config.domain_min) + config.domain_min
    return x


def train_epoch(model, optimizer, config, device):
    model.train()
    optimizer.zero_grad()

    x = sample_collocation_points(config.n_collocation, config, device)
    x.requires_grad_(True)

    physics_loss, energy = compute_physics_loss(model, x, config)
    p_real, p_imag = model(x)
    curl_loss = compute_curl_loss(p_real, p_imag, x)

    loss = config.weight_physics * physics_loss + config.weight_curl * curl_loss
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    return loss.item(), physics_loss.item(), curl_loss.item(), energy.item()


def save_checkpoint(model, optimizer, epoch, loss, energy, checkpoint_dir):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
        'energy': energy,
    }
    path = Path(checkpoint_dir) / f"checkpoint_epoch_{epoch:06d}.pt"
    torch.save(checkpoint, path)
    latest_path = Path(checkpoint_dir) / "checkpoint_latest.pt"
    torch.save(checkpoint, latest_path)
    return path


def load_checkpoint(model, optimizer, checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    return checkpoint['epoch'], checkpoint.get('loss', 0), checkpoint.get('energy', 3.0)

In [ ]:
#=============================================================================
# INITIALIZE MODEL
#=============================================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = LagrangianPINN(config).to(device)
optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

# Check for existing checkpoint to resume
start_epoch = 0
latest_ckpt = Path(CHECKPOINT_DIR) / "checkpoint_latest.pt"
if latest_ckpt.exists():
    print(f"Found checkpoint: {latest_ckpt}")
    start_epoch, last_loss, last_energy = load_checkpoint(model, optimizer, latest_ckpt, device)
    print(f"Resuming from epoch {start_epoch}, loss={last_loss:.6f}, energy={last_energy:.6f}")
    start_epoch += 1
else:
    print("No checkpoint found, starting fresh.")

In [ ]:
#=============================================================================
# TRAINING LOOP
#=============================================================================

print(f"\n{'='*60}")
print(f"Starting training: epochs {start_epoch} to {config.max_epochs}")
print(f"Collocation points: {config.n_collocation:,}")
print(f"Checkpoint interval: {config.checkpoint_interval}")
print(f"{'='*60}\n")

best_energy = float('inf')
training_start = time.time()
history = {'epochs': [], 'loss': [], 'physics_loss': [], 'curl_loss': [], 'energy': []}

try:
    for epoch in range(start_epoch, config.max_epochs):
        epoch_start = time.time()
        loss, physics_loss, curl_loss, energy = train_epoch(model, optimizer, config, device)
        epoch_time = time.time() - epoch_start

        history['epochs'].append(epoch)
        history['loss'].append(loss)
        history['physics_loss'].append(physics_loss)
        history['curl_loss'].append(curl_loss)
        history['energy'].append(energy)

        if energy < best_energy:
            best_energy = energy

        if epoch % config.log_interval == 0:
            elapsed = time.time() - training_start
            print(f"Epoch {epoch:6d} | Loss: {loss:.6f} | Physics: {physics_loss:.6f} | "
                  f"Curl: {curl_loss:.6f} | Energy: {energy:.6f} | Time: {epoch_time:.2f}s | "
                  f"Elapsed: {elapsed/3600:.2f}h")

        if epoch > 0 and epoch % config.checkpoint_interval == 0:
            ckpt_path = save_checkpoint(model, optimizer, epoch, loss, energy, CHECKPOINT_DIR)
            print(f"  -> Saved checkpoint: {ckpt_path.name}")

except KeyboardInterrupt:
    print("\nTraining interrupted by user")
    # Save checkpoint on interrupt
    save_checkpoint(model, optimizer, epoch, loss, energy, CHECKPOINT_DIR)
    print(f"Saved checkpoint at epoch {epoch}")

In [ ]:
#=============================================================================
# RESULTS
#=============================================================================

total_time = time.time() - training_start
final_energy = model.energy.item()

harmonic_zpe = 3.0
expected_energy = 2.97

results = {
    'final_energy': final_energy,
    'best_energy': best_energy,
    'harmonic_zpe': harmonic_zpe,
    'expected_energy': expected_energy,
    'below_harmonic_zpe': final_energy < harmonic_zpe,
    'error_vs_expected': abs(final_energy - expected_energy) / expected_energy * 100,
    'total_epochs': len(history['epochs']),
    'training_time_hours': total_time / 3600,
    'timestamp': datetime.now().isoformat()
}

# Save results
results_path = Path(RESULTS_DIR) / "final_results.json"
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n{'='*60}")
print("TRAINING COMPLETE")
print(f"{'='*60}")
print(f"Final energy:      {final_energy:.6f} a.u.")
print(f"Best energy:       {best_energy:.6f} a.u.")
print(f"Harmonic ZPE:      {harmonic_zpe:.6f} a.u.")
print(f"Expected (lit):    {expected_energy:.6f} a.u.")
print(f"Below harmonic:    {'YES ✓' if final_energy < harmonic_zpe else 'NO ✗'}")
print(f"Error vs expected: {results['error_vs_expected']:.2f}%")
print(f"Training time:     {total_time/3600:.2f} hours")
print(f"Results saved:     {results_path}")
print(f"{'='*60}")

if 2.91 <= final_energy <= 3.03:
    print("\n✓ SUCCESS: Energy within 2% of target (2.97 ± 2%)")
else:
    print(f"\n✗ Target not met: {final_energy:.4f} outside [2.91, 3.03]")

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].semilogy(history['epochs'], history['loss'])
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Total Loss')
axes[0, 0].set_title('Training Loss')
axes[0, 0].grid(True)

axes[0, 1].plot(history['epochs'], history['energy'])
axes[0, 1].axhline(y=3.0, color='r', linestyle='--', label='Harmonic ZPE')
axes[0, 1].axhline(y=2.97, color='g', linestyle='--', label='Expected')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Energy (a.u.)')
axes[0, 1].set_title('Energy Convergence')
axes[0, 1].legend()
axes[0, 1].grid(True)

axes[1, 0].semilogy(history['epochs'], history['physics_loss'])
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Physics Loss')
axes[1, 0].set_title('QHJE Residual')
axes[1, 0].grid(True)

axes[1, 1].semilogy(history['epochs'], history['curl_loss'])
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Curl Loss')
axes[1, 1].set_title('Irrotationality Constraint')
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig(Path(RESULTS_DIR) / 'training_history.png', dpi=150)
plt.show()